Keyword Extraction Using KeyBERT.

Install the relevant packages.

In [1]:
from keybert import KeyBERT

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyBert starts by embedding a document into a vector by turning a chunk of text into a fixed sized vector which represents the semnatics of the document.
Extracts key words using simple techniques - count vectorizer, TFADF vectorizer
Embeds each word using the same model used to embed the document, leaving a list of keyword embeddings. 
Similarity measures will be calculated between document and keyword embeddings. This results in a vector, where each value is a similarity measure between the two. Sorts results in decreasing order to get the most simliar words. 

Adding diversity in the data:
Sometimes there is some redundancy in the data, and repetition. 
Max Sum Similarity
Maximal Marginal Relevance

In [2]:
model = KeyBERT(model = "distilbert-base-nli-mean-tokens")

One document PER Coordinate is needed for analysis

In [3]:
import pandas as pd

In [7]:
all_feedback = pd.read_csv('C:\\Users\\User\\Documents\\GitHub\\proutectapp\\csv-json-files\\joinedfeedback.csv')
ratings = all_feedback.loc[:, ['Latitude', 'Longitude', 'q1', 'q2', 'q3', 'q4']]
responses =  all_feedback.loc[:, ['Latitude', 'Longitude', 'responseList']]

responses

,Latitude,Longitude,responseList
0,52.285500,-1.558030,"The route was busy with the public, but there ..."
1,52.286880,-1.533620,"There was a long, isolated road with a lack of..."
2,52.279250,-1.544650,"There was a long, isolated road with a lack of..."
3,52.290777,-1.533494,The route was busy with the public as there we...
4,52.275950,-1.516430,"Well-lit street, was quite busy with other mem..."
...,...,...,...
779,52.292370,-1.548190,A really nice walk to do in the evening.; The ...
780,52.285970,-1.536280,The route felt fairly safe with some street li...
781,52.282960,-1.545510,The route felt fairly safe with some street li...
782,52.288400,-1.538270,Walking through the park made me feel uneasy a...


Need to classify each coordinate to be safe/unsafe utilising user feedback. Used average of all answers to question 1 "How safe did you feel on the route?" 

In [26]:
classification_list = []
for index, row in ratings.iterrows():
    if row["q1"] > 2.5:
        classification_list.append("safe")
    else:
        classification_list.append("unsafe")

ratings["Classification"] = classification_list

ratings
ratings["Classification"].value_counts()

Classification
safe      413
unsafe    371
Name: count, dtype: int64

In [61]:
ratings['q2'] = ratings['q2'] / 5
ratings['q3'] = ratings['q3'] / 5
ratings['q4'] = ratings['q4'] / 5


In [62]:
ratings

,Latitude,Longitude,q1,q2,q3,q4,Classification
0,52.285500,-1.558030,3.0,0.192,0.032,0.008,safe
1,52.286880,-1.533620,3.0,0.184,0.016,0.008,safe
2,52.279250,-1.544650,3.0,0.184,0.016,0.008,safe
3,52.290777,-1.533494,3.5,0.190,0.032,0.014,safe
4,52.275950,-1.516430,3.0,0.184,0.024,0.008,safe
...,...,...,...,...,...,...,...
779,52.292370,-1.548190,3.0,0.182,0.026,0.016,safe
780,52.285970,-1.536280,2.5,0.176,0.012,0.008,unsafe
781,52.282960,-1.545510,2.5,0.176,0.012,0.008,unsafe
782,52.288400,-1.538270,2.0,0.176,0.008,0.008,unsafe


In [63]:
keywords_list = []

for index, row in responses.iterrows():
  
    doc = row["responseList"]

    keywords = model.extract_keywords(doc, keyphrase_ngram_range=(1,2), stop_words='english', use_mmr=True, diversity=0.9)

    keywords_list.append(keywords)

responses['Key words'] = keywords_list
    

In [64]:
keywords_list = []

for index, row in responses.iterrows():
  
    doc = row["responseList"]

    keywords = model.extract_keywords(doc, keyphrase_ngram_range=(1,3), stop_words='english', use_mmr=True, diversity=0.9)

    keywords_list.append(keywords)

responses['Key words 3'] = keywords_list

In [65]:
responses

,Latitude,Longitude,responseList,Key words,Key words 2,Key words 3
0,52.285500,-1.558030,"The route was busy with the public, but there ...","[(busy public, 0.5604), (lot cars, 0.3592), (u...","[(busy, 0.3785), (unsafe, 0.3211), (cars, 0.31...","[(cars feel unsafe, 0.7099), (busy public, 0.5..."
1,52.286880,-1.533620,"There was a long, isolated road with a lack of...","[(isolated road, 0.7717), (long isolated, 0.65...","[(isolated, 0.57), (road, 0.5312), (long, 0.45...","[(long isolated road, 0.8514), (lack lighting ..."
2,52.279250,-1.544650,"There was a long, isolated road with a lack of...","[(isolated road, 0.7717), (long isolated, 0.65...","[(isolated, 0.57), (road, 0.5312), (long, 0.45...","[(long isolated road, 0.8514), (lack lighting ..."
3,52.290777,-1.533494,The route was busy with the public as there we...,"[(parade safer, 0.4993), (shops needed, 0.3651...","[(parade, 0.3846), (shops, 0.2816), (nearby, 0...","[(busy public shops, 0.5989), (parade, 0.3846)..."
4,52.275950,-1.516430,"Well-lit street, was quite busy with other mem...","[(lit street, 0.7497), (busy members, 0.6292),...","[(street, 0.5627), (public, 0.5542), (busy, 0....","[(public lit street, 0.7701), (street quite bu..."
...,...,...,...,...,...,...
779,52.292370,-1.548190,A really nice walk to do in the evening.; The ...,"[(night drunk, 0.5259), (short sweet, 0.2917),...","[(drunk, 0.3404), (short, 0.2023), (12am, 0.13...","[(park night drunk, 0.5805), (poorly lit 12am,..."
780,52.285970,-1.536280,The route felt fairly safe with some street li...,"[(felt helpless, 0.381), (night route, 0.3066)...","[(helpless, 0.3453), (night, 0.2396), (park, 0...","[(park felt helpless, 0.581), (night route, 0...."
781,52.282960,-1.545510,The route felt fairly safe with some street li...,"[(felt helpless, 0.381), (night route, 0.3066)...","[(helpless, 0.3453), (night, 0.2396), (park, 0...","[(park felt helpless, 0.581), (night route, 0...."
782,52.288400,-1.538270,Walking through the park made me feel uneasy a...,"[(uneasy lighting, 0.5117), (walking park, 0.4...","[(uneasy, 0.404), (walking, 0.3186), (park, 0....","[(park feel uneasy, 0.6819), (uneasy lighting,..."


In [66]:
combined_df = pd.merge(ratings, responses, on=["Latitude", "Longitude"])

In [67]:
combined_df

,Latitude,Longitude,q1,q2,q3,q4,Classification,responseList,Key words,Key words 2,Key words 3
0,52.285500,-1.558030,3.0,0.192,0.032,0.008,safe,"The route was busy with the public, but there ...","[(busy public, 0.5604), (lot cars, 0.3592), (u...","[(busy, 0.3785), (unsafe, 0.3211), (cars, 0.31...","[(cars feel unsafe, 0.7099), (busy public, 0.5..."
1,52.286880,-1.533620,3.0,0.184,0.016,0.008,safe,"There was a long, isolated road with a lack of...","[(isolated road, 0.7717), (long isolated, 0.65...","[(isolated, 0.57), (road, 0.5312), (long, 0.45...","[(long isolated road, 0.8514), (lack lighting ..."
2,52.279250,-1.544650,3.0,0.184,0.016,0.008,safe,"There was a long, isolated road with a lack of...","[(isolated road, 0.7717), (long isolated, 0.65...","[(isolated, 0.57), (road, 0.5312), (long, 0.45...","[(long isolated road, 0.8514), (lack lighting ..."
3,52.290777,-1.533494,3.5,0.190,0.032,0.014,safe,The route was busy with the public as there we...,"[(parade safer, 0.4993), (shops needed, 0.3651...","[(parade, 0.3846), (shops, 0.2816), (nearby, 0...","[(busy public shops, 0.5989), (parade, 0.3846)..."
4,52.275950,-1.516430,3.0,0.184,0.024,0.008,safe,"Well-lit street, was quite busy with other mem...","[(lit street, 0.7497), (busy members, 0.6292),...","[(street, 0.5627), (public, 0.5542), (busy, 0....","[(public lit street, 0.7701), (street quite bu..."
...,...,...,...,...,...,...,...,...,...,...,...
779,52.292370,-1.548190,3.0,0.182,0.026,0.016,safe,A really nice walk to do in the evening.; The ...,"[(night drunk, 0.5259), (short sweet, 0.2917),...","[(drunk, 0.3404), (short, 0.2023), (12am, 0.13...","[(park night drunk, 0.5805), (poorly lit 12am,..."
780,52.285970,-1.536280,2.5,0.176,0.012,0.008,unsafe,The route felt fairly safe with some street li...,"[(felt helpless, 0.381), (night route, 0.3066)...","[(helpless, 0.3453), (night, 0.2396), (park, 0...","[(park felt helpless, 0.581), (night route, 0...."
781,52.282960,-1.545510,2.5,0.176,0.012,0.008,unsafe,The route felt fairly safe with some street li...,"[(felt helpless, 0.381), (night route, 0.3066)...","[(helpless, 0.3453), (night, 0.2396), (park, 0...","[(park felt helpless, 0.581), (night route, 0...."
782,52.288400,-1.538270,2.0,0.176,0.008,0.008,unsafe,Walking through the park made me feel uneasy a...,"[(uneasy lighting, 0.5117), (walking park, 0.4...","[(uneasy, 0.404), (walking, 0.3186), (park, 0....","[(park feel uneasy, 0.6819), (uneasy lighting,..."


In [68]:
from collections import Counter

In [69]:
safe_keywords = combined_df[combined_df["Classification"] == "safe"]["Key words 3"].explode()
unsafe_keywords = combined_df[combined_df["Classification"] == "unsafe"]["Key words 3"].explode()

print(safe_keywords)

print(unsafe_keywords)

safe_word_count = Counter(safe_keywords)
unsafe_word_count = Counter(unsafe_keywords)

top_safe_words = [keyword for keyword, count in safe_word_count.most_common(10)]
top_unsafe_words = [keyword for keyword, count in unsafe_word_count.most_common(10)]

0      (cars feel unsafe, 0.7099)
0           (busy public, 0.5604)
0              (lot cars, 0.3592)
0                (unsafe, 0.3211)
0           (times route, 0.0321)
                  ...            
779    (park night drunk, 0.5805)
779      (poorly lit 12am, 0.564)
779      (12am don enjoy, 0.2233)
779             (walking, 0.0672)
779                  (park, 0.06)
Name: Key words 3, Length: 2065, dtype: object
6      (unsafe residential access, 0.7517)
6           (street lighting felt, 0.3424)
6                    (road street, 0.3144)
6                    (residential, 0.2465)
6                  (access public, 0.1257)
                      ...                 
783             (park night drunk, 0.7046)
783           (walking park night, 0.5303)
783                  (people dark, 0.4086)
783                          (park, 0.132)
783                       (walking, 0.118)
Name: Key words 3, Length: 1855, dtype: object


In [70]:
print(top_safe_words)

print(top_unsafe_words)

[('cars feel unsafe', 0.7099), ('busy public', 0.5604), ('lot cars', 0.3592), ('unsafe', 0.3211), ('times route', 0.0321), ('walk park nice', 0.7759), ('daytime felt safe', 0.6322), ('park', 0.4643), ('busy daytime', 0.3986), ('route', 0.0843)]
[('park felt helpless', 0.581), ('night route', 0.3066), ('street lighting time', 0.2877), ('felt fairly safe', 0.2228), ('park', 0.1952), ('away shops public', 0.5348), ('overall preferred busy', 0.3259), ('hospital nearby', 0.2667), ('safe', 0.2519), ('spaces hospital', 0.2223)]


following code is to alter safety score based on presence of words:

In [71]:
# def adjust_safety_score(row, pos_keywords, neg_keywords, adjustment_factor=0.1):
#     # Assuming the initial safety score is in a column named 'safety_score'
#     initial_score = row['safety_score']
#     keywords = row['keywords_list']
    
#     # Check for the presence of top positive/negative keywords
#     pos_presence = any(keyword in keywords for keyword in pos_keywords)
#     neg_presence = any(keyword in keywords for keyword in neg_keywords)
    
#     # Adjust the score based on keyword presence
#     if pos_presence:
#         return initial_score * (1 + adjustment_factor)  # Increase by adjustment_factor
#     elif neg_presence:
#         return initial_score * (1 - adjustment_factor)  # Decrease by adjustment_factor
#     else:
#         return initial_score  # No change

# # Apply the adjustment function to each row
# joinedfeedback['adjusted_safety_score'] = joinedfeedback.apply(adjust_safety_score, args=(top_positive_keywords, top_negative_keywords), axis=1)


Getting crime data for this dataframe - creating new safety scores. 

In [86]:
crime_severity_scores = {
    "Drugs" : 2,
    "Shoplifting" : 2,
    "Burglary" : 5,
    "Theft from the person" : 8,
    "Possession of weapons" : 9,
    "Violence and sexual offences" : 10,
    "Criminal damage and arson" : 7,
    "Vehicle crime" : 4,
    "Bicycle theft " : 2,
    "Other theft" : 1,
    "Public order" : 1,
    "Robbery" : 5, 
    "Anti-social behaviour" : 5,
    "Other crime" : 1
}

In [87]:
import numpy as np 

In [88]:
def get_lat_lon_ranges(lat, lon, radius):
    # Earth's radius in meters
    EARTH_RADIUS = 6378137

    # Convert latitude and longitude from degrees to radians
    lat_rad = np.radians(lat)

    # Calculate deltas
    dLat = radius/EARTH_RADIUS
    dLon = radius/(EARTH_RADIUS * np.cos(lat_rad))

    # Convert deltas from radians to degrees
    dLat_deg = np.degrees(dLat)
    dLon_deg = np.degrees(dLon)

    # Define ranges
    lat_range = (lat - dLat_deg, lat + dLat_deg)
    lon_range = (lon - dLon_deg, lon + dLon_deg)

    return lat_range, lon_range

In [89]:
def get_crimes_in_range(all_crime_data, long_min, long_max, lat_min, lat_max, crime_severity_scores):
    # Filter the data to only include crimes within the specified ranges
    crimes_in_range = all_crime_data[
        (all_crime_data['Longitude'] >= long_min) & 
        (all_crime_data['Longitude'] <= long_max) & 
        (all_crime_data['Latitude'] >= lat_min) & 
        (all_crime_data['Latitude'] <= lat_max)
    ]

    crimes_in_range = crimes_in_range.copy()
    
    # Add severity score for each crime
    crimes_in_range['Severity Score'] = crimes_in_range['Crime type'].map(crime_severity_scores)
    # print(crimes_in_range)
    # Calculate average severity score for the crimes in the range
    
    if not crimes_in_range.empty:
        average_severity_score = crimes_in_range['Severity Score'].mean()
        print(average_severity_score)
    else:
        average_severity_score = 0

    return len(crimes_in_range), average_severity_score

In [90]:
crime_data_2021 = pd.read_csv("C:\\Users\\User\\Documents\\GitHub\\proutectapp\\csv-json-files\\crimes_in_range_2021.csv")
crime_data_2022 = pd.read_csv("C:\\Users\\User\\Documents\\GitHub\\proutectapp\\csv-json-files\\crimes_in_range_2022.csv")
crime_data_2023 = pd.read_csv("C:\\Users\\User\\Documents\\GitHub\\proutectapp\\csv-json-files\\crimes_in_range_2023.csv")

In [98]:
def get_coords(dataframe):
    all_locations = [f"{row['Latitude']}, {row['Longitude']}" for index, row in dataframe.iterrows()]
    return [(float(lat), float(lon)) for lat, lon in (s.split(', ') for s in all_locations)]

coords21 = get_coords(crime_data_2021)
coords22 = get_coords(crime_data_2022)
coords23 = get_coords(crime_data_2023)

In [104]:
# Gets a dictionary of all crimes, public space data for each coordinate in a 350m radius 
def create_dict(all_crime_data, all_coordinates, crime_severity_scores):
    coordinate_dict = {}
    for each in all_coordinates:
        coordinate_dict[each] = [0, 0]
        # Gets lat/lon range within a 50m radius
        lat_range, lon_range = get_lat_lon_ranges(each[0], each[1], 50)
        coordinate_dict[each][0], coordinate_dict[each][1] = get_crimes_in_range(all_crime_data, lon_range[0], lon_range[1], lat_range[0], lat_range[1], crime_severity_scores)
    return coordinate_dict

In [105]:
data_2021 = create_dict(crime_data_2021, coords21, crime_severity_scores)
data_2022 = create_dict(crime_data_2021, coords22, crime_severity_scores)
data_2023 = create_dict(crime_data_2021, coords23, crime_severity_scores)

5.0
4.5
5.0
3.857142857142857
5.769230769230769
6.532467532467533
5.40625
6.637931034482759
6.637931034482759
5.225806451612903
6.944444444444445
3.3366336633663365
6.231884057971015
8.084745762711865
5.3
6.6
6.029197080291971
5.368421052631579
8.084745762711865
6.231884057971015
5.368421052631579
3.0526315789473686
3.2419354838709675
6.029197080291971
3.2419354838709675
6.029197080291971
3.3366336633663365
6.029197080291971
4.863636363636363
4.863636363636363
6.029197080291971
3.3366336633663365
3.0526315789473686
7.222222222222222
5.368421052631579
4.0
4.92
5.46969696969697
3.2419354838709675
5.368421052631579
6.231884057971015
6.231884057971015
6.026315789473684
3.0526315789473686
5.9523809523809526
6.285714285714286
6.026315789473684
6.285714285714286
6.285714285714286
6.934210526315789
6.934210526315789
5.125
6.934210526315789
5.125
6.934210526315789
6.838709677419355
5.125
6.838709677419355
6.838709677419355
5.7073170731707314
7.0
7.0
7.0
5.7073170731707314
5.2
5.2
5.8
3.13636363

In [107]:
data_2021

{(52.279611, -1.501874): [9, 5.0],
 (52.272738, -1.498624): [2, 4.5],
 (52.276068, -1.499642): [7, 3.857142857142857],
 (52.294499, -1.530402): [13, 5.769230769230769],
 (52.293096, -1.530358): [79, 6.532467532467533],
 (52.294583, -1.529037): [32, 5.40625],
 (52.292775, -1.530962): [60, 6.637931034482759],
 (52.292862, -1.534759): [31, 5.225806451612903],
 (52.292301, -1.529428): [18, 6.944444444444445],
 (52.291967, -1.535736): [103, 3.3366336633663365],
 (52.288299, -1.535877): [69, 6.231884057971015],
 (52.293131, -1.536853): [59, 8.084745762711865],
 (52.29137, -1.532561): [11, 5.3],
 (52.289332, -1.537743): [5, 6.6],
 (52.292609, -1.536712): [151, 6.029197080291971],
 (52.289063, -1.535854): [23, 5.368421052631579],
 (52.290783, -1.536394): [76, 3.0526315789473686],
 (52.291278, -1.534248): [63, 3.2419354838709675],
 (52.290069, -1.535477): [67, 4.863636363636363],
 (52.289549, -1.531436): [19, 7.222222222222222],
 (52.294878, -1.535383): [1, 4.0],
 (52.289819, -1.535832): [76, 4

Defines the weightings used to find the safety score.

In [108]:
# Original weights based on percentages
survey_weightings = {
    "crime": 95.9,
    "public_space": (75 + 83.3) / 2,  
    "emergency": 80.2,
    "crowded": 78.2,
    "lighting": 85.4,
    "pos_keywords": 10, # Gave a percentage of 10 to start with. 
    "neg_keywords": -10 # Gave a percentage of 10 to start with. 
}

# Sum of the original weights
total_weight = sum(survey_weightings.values())

# Normalize each weight
normalized_weights = {k: v / total_weight for k, v in survey_weightings.items()}

print(normalized_weights)

{'crime': 0.2289602482989137, 'public_space': 0.18896979825713262, 'emergency': 0.19147666228960247, 'crowded': 0.18670168318013608, 'lighting': 0.20389160797421513, 'pos_keywords': 0.02387489554733198, 'neg_keywords': -0.02387489554733198}


In [109]:
# Function to calculate the safety score of a particular coordinate
def calculate_safety_score(max_crime, num_crime, space_rating, emergency_rating, crowd_rating, 
                           unlit_rating, severity, weightings, keywords, pos_keywords, neg_keywords):
    
    safety_score = 0

    if num_crime > 0:
        safety_score += ((max_crime - num_crime) / max_crime) * weightings["crime"] * ((10 - severity) / 10)
    else:
        safety_score += weightings["crime"]

    # Public space score calculation
    if space_rating > 0:
        safety_score += (space_rating * weightings["public_space"])
    else: 
        safety_score -= weightings["public_space"]
    
    if emergency_rating > 0:
        safety_score += (emergency_rating * weightings["emergency"])
    else: 
        safety_score -= weightings["emergency"]

    if crowd_rating > 0:
        safety_score += (crowd_rating * weightings["crowded"])
    else: 
        safety_score -= weightings["crowded"]

    if unlit_rating > 0:
        safety_score += (unlit_rating * weightings["lighting"])
    else: 
        safety_score -= weightings["lighting"]

    pos_count = sum(keywords.count(word) for word in pos_keywords)
    neg_count = sum(keywords.count(word) for word in neg_keywords)

    total_keywords = pos_count + neg_count

    if total_keywords > 0:
        pos_ratio = pos_count / total_keywords
        neg_ratio = neg_count / total_keywords
        safety_score += (pos_ratio * weightings["pos_keywords"]) + (neg_ratio * weightings["neg_keywords"])
        
    return safety_score
    

In [ ]:
def get_safety_scores(coords, max_crime, num_crime, space_rating, emergency_rating, 
                      crowd_rating, unlit_rating, severity, weightings, keywords, pos_keywords, 
                      neg_keywords, year, output_file):
    safety_scores = []
    for coordinate, (crime_num, severity) in coords.items():
        lat = coordinate[0]
        lon = coordinate[1]

        safety_score = calculate_safety_score(max_crime, num_crime, space_rating, emergency_rating, 
                                              crowd_rating, unlit_rating, severity, weightings, keywords, 
                                              pos_keywords, neg_keywords)
        
        safety_scores.append({
            "Latitude": lat,
            "Longitude": lon,
            "Safety Score": safety_score,
            "Year": year
        })

    safety_df = pd.DataFrame(safety_scores)
    existing_df = pd.read_csv(output_file)
    joined_df = pd.concat([existing_df, safety_df], ignore_index=True)
    joined_df.to_csv(output_file, index=False)

Removing repetition - Max_sum similarity and Maximal Marginal Relevance.

In [83]:
# keywords = model.extract_keywords(doc)
# model.extract_keywords(doc, keyphrase_ngram_range=(1,3), stop_words='english',
#                        use_maxsum=True, nr_candidates=20, top_n=4)
# model.extract_keywords(doc, keyphrase_ngram_range=(1,3), stop_words='english',
#                        use_mmr=True, diversity=0.9)